# Día 3 · Unidad 3 — List comprehensions y dict comprehensions

**Objetivos de aprendizaje**
- Sustituir un `for` que construye una lista por una *list comprehension* equivalente.
- Añadir condiciones (`if`, e `if/else`) dentro de una comprehension.
- Construir diccionarios en una línea con *dict comprehension*.
- Aplicar comprehensions a casos reales de una cartera de pólizas: filtrar, transformar y reindexar.
- Conocer `eval()` y por qué **nunca** debe usarse sobre una entrada que no controlas.

**Duración estimada:** 45 minutos.

**Requisitos previos:** `03_teoria_bucles_for.ipynb` de este mismo día.

## 1. De un `for` a una *list comprehension*

Ya has escrito varias veces el patrón "lista vacía + `for` + `.append()`". Es tan frecuente que Python le da una sintaxis propia, más compacta: la ***list comprehension***.

```python
[<expresión> for <elemento> in <iterable>]
```

Vamos a partir de una cartera de pólizas y comparar ambas formas.

In [ ]:
polizas = [
    {"id": "P001", "tipo": "auto", "prima_anual": 480.50, "siniestros": 0, "antiguedad_anios": 3},
    {"id": "P002", "tipo": "hogar", "prima_anual": 210.00, "siniestros": 2, "antiguedad_anios": 1},
    {"id": "P003", "tipo": "vida", "prima_anual": 650.00, "siniestros": 0, "antiguedad_anios": 7},
    {"id": "P004", "tipo": "auto", "prima_anual": 395.75, "siniestros": 1, "antiguedad_anios": 5},
    {"id": "P005", "tipo": "hogar", "prima_anual": 180.00, "siniestros": 0, "antiguedad_anios": 2},
]

# Forma con for + append: extraer solo los IDs de póliza
ids_v1 = []
for p in polizas:
    ids_v1.append(p["id"])
print(ids_v1)

# La misma operación como list comprehension
ids_v2 = [p["id"] for p in polizas]
print(ids_v2)


Ambas dan el mismo resultado. La comprehension no es "magia": es exactamente el mismo bucle, escrito de forma más compacta y, una vez te acostumbras a leerla, más clara sobre la intención ("estoy construyendo una lista a partir de otra cosa").

## 2. Comprehensions con condición: filtrar

Añadiendo un `if` al final se filtran los elementos antes de incluirlos — el equivalente a un Autofiltro con condición aplicado durante la propia construcción de la lista.

```python
[<expresión> for <elemento> in <iterable> if <condición>]
```

In [ ]:
# Solo los IDs de las pólizas de auto
ids_auto = [p["id"] for p in polizas if p["tipo"] == "auto"]
print("Pólizas de auto:", ids_auto)

# Solo las pólizas con algún siniestro registrado (aquí nos quedamos con el dict completo, no solo el id)
polizas_con_siniestros = [p for p in polizas if p["siniestros"] > 0]
print("Pólizas con siniestros:", [p["id"] for p in polizas_con_siniestros])


> 📊 **En Excel esto sería...** un Autofiltro (o `FILTRAR()` en Excel moderno) aplicado a una tabla: te quedas solo con las filas que cumplen la condición, sin tocar la tabla original.

## 3. Comprehensions con transformación

No hace falta que la expresión sea el elemento tal cual: puede ser cualquier cálculo sobre él. Es la parte de "transformar" del filtrado anterior.

In [ ]:
# Aplicar un recargo del 5% a la prima de las pólizas de auto con algún siniestro
primas_con_recargo = [
    round(p["prima_anual"] * 1.05, 2)
    for p in polizas
    if p["tipo"] == "auto" and p["siniestros"] > 0
]
print("Primas con recargo aplicado:", primas_con_recargo)


También se puede incluir un `if/else` **dentro** de la expresión (no al final) cuando quieres transformar el elemento de una forma u otra según la condición, en vez de descartarlo:

```python
[<expr_si_cumple> if <condición> else <expr_si_no> for <elemento> in <iterable>]
```

Es la expresión condicional del primer notebook del día, metida dentro de una comprehension.

In [ ]:
# Etiquetar cada póliza como "revisar" o "ok" según tenga siniestros o no
# (fíjate: aquí el if/else va PEGADO a la expresión, no al final -- no filtra, transforma)
estado_revision = [
    "revisar" if p["siniestros"] > 0 else "ok"
    for p in polizas
]
print(list(zip([p["id"] for p in polizas], estado_revision)))


> ⚠️ **Error típico**: confundir las dos formas de usar `if` en una comprehension. `[x for x in lst if cond]` **filtra** (algunos elementos no aparecen en el resultado). `[x if cond else y for x in lst]` **no filtra, transforma** — el resultado tiene siempre el mismo número de elementos que el iterable original. Mezclarlas mentalmente es una fuente frecuente de listas con menos (o más) elementos de los esperados, sin que Python avise con ningún error: es un fallo de lógica silencioso, de los que hay que detectar comprobando la longitud del resultado.

In [ ]:
print("Original:      ", len(polizas), "pólizas")
print("Filtrado (auto):", len([p for p in polizas if p["tipo"] == "auto"]), "pólizas")
print("Transformado:   ", len(estado_revision), "elementos (igual que el original, porque no filtra)")


## 4. *Dict comprehension*

La misma idea, para construir diccionarios en una línea:

```python
{<expresión_llave>: <expresión_valor> for <elemento> in <iterable>}
```

In [ ]:
# Diccionario id_poliza -> prima_anual
primas_por_id = {p["id"]: p["prima_anual"] for p in polizas}
print(primas_por_id)


In [ ]:
# Con condición: solo las pólizas de hogar, id -> número de siniestros
siniestros_hogar = {p["id"]: p["siniestros"] for p in polizas if p["tipo"] == "hogar"}
print(siniestros_hogar)


In [ ]:
# Invertir un diccionario (llave <-> valor) -- útil cuando cada valor es único
provincia_por_id = {"P001": "BCN", "P002": "GIR", "P003": "TAR"}
id_por_provincia = {provincia: id_poliza for id_poliza, provincia in provincia_por_id.items()}
print(id_por_provincia)


**Para ti:** con `polizas`, construye un diccionario `prima_media_por_tipo` que mapee cada `tipo` de póliza (`"auto"`, `"hogar"`, `"vida"`) a la prima media de ese tipo, redondeada a 2 decimales. Pista: necesitarás una comprehension (o un bucle) por cada tipo distinto — no tiene por qué salir en una única línea perfecta a la primera; que funcione es más importante que lo compacto que sea.

## 5. `eval()`: qué es y por qué casi nunca deberías usarlo

`eval()` toma una cadena de texto y la ejecuta como si fuera código Python, devolviendo el resultado.

In [ ]:
print(eval("2 + 4"))

x = 100
print(eval("x + 1"))  # eval() puede ver las variables ya definidas en el entorno actual


> ⚠️ **Nota de seguridad — muy importante**: `eval()` ejecuta **cualquier** código Python que reciba como texto, no solo expresiones aritméticas inocentes. Si ese texto viniera de una fuente que no controlas del todo (un `input()` de un usuario, un campo de un formulario, el contenido de un archivo subido por un tercero...), `eval()` le está dando a esa fuente la capacidad de ejecutar código arbitrario en tu máquina — desde borrar archivos hasta leer datos sensibles.
>
> **Regla de este curso y de tu trabajo profesional: nunca uses `eval()` sobre una entrada que no controlas por completo.** Si necesitas evaluar una expresión matemática que sí escribe un usuario, existen alternativas mucho más seguras (por ejemplo, la librería `ast.literal_eval` para literales simples, o un parser matemático dedicado) que no ejecutan código arbitrario.

In [ ]:
import ast

# ast.literal_eval SOLO interpreta literales de Python (números, strings, listas, dicts...),
# nunca ejecuta código ni llama a funciones -- es la alternativa segura cuando el texto
# viene de fuera y solo esperas un dato, no una instrucción.
texto_recibido = "[1, 2, 3]"
valores = ast.literal_eval(texto_recibido)
print(valores, type(valores))

try:
    ast.literal_eval("__import__('os').listdir('.')")  # intento de ejecutar código -- se rechaza
except (ValueError, SyntaxError) as e:
    print(f"Rechazado de forma segura: {type(e).__name__}")


## Resumen

- `[<expr> for <elem> in <iterable>]` es la forma compacta de un `for` que construye una lista con `.append()` — el mismo bucle, otra sintaxis.
- Añadir `if <condición>` al final **filtra**: algunos elementos del iterable original no aparecen en el resultado.
- Poner `<expr_A> if <condición> else <expr_B>` **antes** del `for` **transforma**: el resultado tiene siempre el mismo número de elementos que el original. Confundir ambas formas es un fallo de lógica que no lanza ningún error — se detecta comprobando la longitud del resultado.
- `{<llave>: <valor> for <elem> in <iterable>}` construye un diccionario con la misma lógica.
- Las comprehensions son ideales para filtrar y transformar carteras de pólizas, primas o siniestros en una línea, sin sacrificar legibilidad si no se anidan en exceso.
- `eval()` ejecuta código Python arbitrario a partir de una cadena de texto — **nunca debe usarse sobre una entrada que no controlas completamente** (input de usuario, archivo externo, etc.); para literales simples, usa `ast.literal_eval`.

**Siguiente notebook:** `05_teoria_errores_y_debugging.ipynb` — leer un traceback de arriba abajo y usar Claude Code para depurar con criterio.